### tool_choice 的使用

#### 1. tool_choice 的取值为 none

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [2]:
# 定义工具
@tool(
        "get_weacher_and_forecast", 
        parse_docstring=True
      )
def get_weather(city:str, if_forecast:bool):
    """
        查询当日的天气, 可以包含明天的天气预报

        Args:
            city: 城市名称
            if_forecast: 是否包含明天天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))
get_weather_info = convert_to_openai_tool(get_weather)
rprint(type(get_weather_info))

{
    'type': 'function',
    'function': {
        'name': 'get_weacher_and_forecast',
        'description': '查询当日的天气, 可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称', 'type': 'string'},
                'if_forecast': {'description': '是否包含明天天气', 'type': 'boolean'}
            },
            'required': ['city', 'if_forecast'],
            'type': 'object'
        }
    }
}

<class 'dict'>

In [4]:
# 1. 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather], tool_choice="none")

# 2. 维护一个消息列表
messages = [
    HumanMessage("今天杭州的天气怎么样? 明天呢?")
]

# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke(messages)
rprint(response)

AIMessage(
    content='很抱歉，我无法获取实时天气信息，因为我没有联网功能，也不知道当前的准确日期。因此没办法告诉你杭州今天和
明天的天气情况。\n\n建议你可以通过以下方式查询：\n\n- **手机自带天气应用**：查看实时天气和未来几天的预报\n- 
**中国天气网**(weather.com.cn)：中国气象局官方网站\n- **墨迹天气、彩云天气**等专业天气App\n- 
**搜索引擎**：直接搜索“杭州天气”\n- 
**语音助手**：如小爱同学、Siri、小度等\n\n杭州属亚热带季风气候，四季分明。如果你告诉我出行的大致季节，我可以给你一
些穿衣或出行的建议。还有什么我可以帮你的吗？',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 334,
            'prompt_tokens': 22,
            'total_tokens': 356,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 186,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '2026111113184476497f5a034c4592',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1c90b-50d2-7b53-8af1-c68497372b98-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 22,
        'output_tokens': 334,
        'total_tokens': 356,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 186}
    }
)

#### 2. tool_choice 取值为 auto

In [5]:
# 1. 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather], tool_choice="auto")

# 2. 维护一个消息列表
messages = [
    HumanMessage("今天杭州的天气怎么样? 明天呢?")
]

# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke(messages)
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 73,
            'prompt_tokens': 204,
            'total_tokens': 277,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 49,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '202611111319480c03a400a6c141d6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1c90c-4977-7992-baf7-3487ee971004-0',
    tool_calls=[
        {
            'name': 'get_weacher_and_forecast',
            'args': {'city': '杭州', 'if_forecast': True},
            'id': 'call_-7086077872347014278',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 204,
        'output_tokens': 73,
        'total_tokens': 277,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 49}
    }
)

#### 3. tool_choice 取值为 required

In [7]:
# 1. 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather], tool_choice="required")

# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke("2+3=?")
rprint(response)

AIMessage(
    content='2+3=5\n\n这是一个简单的加法运算，不需要查询天气等工具。如果您有其他问题，欢迎继续提问！',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 64,
            'prompt_tokens': 198,
            'total_tokens': 262,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 35,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '202611111321525301c4794c6b4d6c',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1c90e-2d93-7ff2-95f2-da6a8b99f9a4-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 198,
        'output_tokens': 64,
        'total_tokens': 262,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 35}
    }
)

#### 4. 强制调用指定工具

In [8]:
# 1. 定义工具
# 定义股票查询工具
@tool('aaa', parse_docstring=True)
def get_stock_price(company:str, timeframe:str = "today") -> str:
    """
    获取指定公司的股票价格信息

    Args:
        company: 公司名称(如: 苹果, 微软, 谷歌)
        timeframe: 时间范围(today-今天, week-本周, month-本月)
    """
    # 模拟股票数据
    mock_data = {
        "苹果": {"today": 185.20, "week": 183.40, "month": 180.75},
        "微软": {"today": 7666.20, "week": 666.40, "month": 888.75},
        "谷歌": {"today": 23422.20, "week": 888.40, "month": 777.75}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe} 价格: {price}美元"
    else:
        return f"未找到股票代码{company}的数据"

# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称
    """
    # 模拟新闻数据
    mock_news = {
        "苹果": [
            "苹果哈哈",
            "苹果"
        ],
        "微软": [
            "微软"
        ],
        "谷歌": [
            "google"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# 2. 初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools, tool_choice="search_news")


# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke("2+3=?")
rprint(response)

AIMessage(
    content='2+3=5',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 66,
            'prompt_tokens': 164,
            'total_tokens': 230,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 59,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '202611111330002672d9ccc90f40af',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1c915-9eb0-7b63-97df-d619d0b0043d-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 164,
        'output_tokens': 66,
        'total_tokens': 230,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 59}
    }
)